In [13]:
import math
from typing import NamedTuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
class TriangleSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, gamma: float):
        ctx.save_for_backward(x)
        ctx.gamma = gamma
        return (x >= 0.0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        (x,) = ctx.saved_tensors
        grad_x = grad_output * ctx.gamma * torch.clamp(1.0 - x.abs(), min=0.0)
        return grad_x, None

def spike_fn(x: torch.Tensor, gamma: float = 0.3) -> torch.Tensor:
    return TriangleSpike.apply(x, gamma)

In [15]:
class LIState(NamedTuple):
    vs: torch.Tensor


class LIFState(NamedTuple):
    vs: torch.Tensor
    z: torch.Tensor

In [16]:
class BaseSpikingCell(nn.Module):
    def __init__(self, threshold: float = 1.0, dt: float = 1.0, gamma: float = 0.3):
        super().__init__()
        self.thr = threshold
        self.dt = dt
        self.gamma = gamma

    def spike_and_reset(self, vs_t: torch.Tensor, prev_z: torch.Tensor, thr: Union[float, torch.Tensor]):
        vs_t = vs_t - prev_z * thr
        z_t = spike_fn((vs_t - thr) / thr, self.gamma)
        return vs_t, z_t

In [17]:
class BaseLayer(nn.Module):
    def __init__(self, cell: nn.Module):
        super().__init__()
        self.cell = cell

    def forward(self, input_tensor: torch.Tensor) -> torch.Tensor:
        batch_size = input_tensor.shape[0]
        state = self.cell.init_states(batch_size, input_tensor.device)
        outputs = []
        for x_t in input_tensor.unbind(1):
            out_t, state = self.cell(x_t, state)
            outputs.append(out_t)
        return torch.stack(outputs, dim=1)

In [18]:
class LICell(nn.Module):
    def __init__(self, in_features: int, out_features: int, tau_soma: float = 20.0, dt: float = 1.0):
        super().__init__()
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.register_buffer("decay_soma", torch.tensor(math.exp(-dt / tau_soma)))

    def init_states(self, batch_size: int, device: torch.device) -> LIState:
        return LIState(vs=torch.zeros(batch_size, self.out_features, device=device))

    def forward(self, x: torch.Tensor, state: LIState) -> tuple[torch.Tensor, LIState]:
        is_t = F.linear(x, self.weight, self.bias)
        vs_t = self.decay_soma * state.vs + (1.0 - self.decay_soma) * is_t
        return vs_t, LIState(vs=vs_t)

In [19]:
class LIFCell(BaseSpikingCell):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        tau_soma: float = 20.0,
        threshold: float = 1.0,
        dt: float = 1.0,
        gamma: float = 0.3,
    ):
        super().__init__(threshold=threshold, dt=dt, gamma=gamma)
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.rec_weight = nn.Parameter(torch.empty(out_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.rec_weight, a=math.sqrt(5))
        self.register_buffer("decay_soma", torch.tensor(math.exp(-dt / tau_soma)))

    def init_states(self, batch_size: int, device: torch.device) -> LIFState:
        shape = (batch_size, self.out_features)
        return LIFState(
            vs=torch.zeros(shape, device=device),
            z=torch.zeros(shape, device=device),
        )

    def forward(self, x: torch.Tensor, state: LIFState) -> tuple[torch.Tensor, LIFState]:
        is_t = F.linear(x, self.weight, self.bias) + F.linear(state.z, self.rec_weight)
        vs_t = self.decay_soma * state.vs + (1.0 - self.decay_soma) * is_t
        vs_t, z_t = self.spike_and_reset(vs_t, state.z, self.thr)
        return z_t, LIFState(vs=vs_t, z=z_t)

In [20]:
class LIFLayer(BaseLayer):
    def __init__(self, in_features: int, out_features: int, **kwargs):
        super().__init__(LIFCell(in_features, out_features, **kwargs))

class LILayer(BaseLayer):
    def __init__(self, in_features: int, out_features: int, **kwargs):
        super().__init__(LICell(in_features, out_features, **kwargs))

In [21]:
class Simple_RSNN(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 200,
        output_size: int = 1,
        ctx_size: int = 0,
        tau_soma: float = 20.0,
        tau_beta: float = 200.0,
    ):
        super().__init__()
        total_in = input_size + ctx_size
        self.l1 = LIFLayer(total_in, hidden_size, tau_soma=tau_soma)
        self.l2 = LILayer(hidden_size, output_size, tau_soma=tau_soma)

    def forward(self, inputs: Union[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
        if isinstance(inputs, (tuple, list)):
            x, c = inputs
            x = torch.cat([x, c], dim=-1)
        else:
            x = inputs
        return self.l2(self.l1(x))

In [ ]:
def generate_hard_tmaze_batch(batch_size=64, cue_steps=60, delay_steps=50, decision_steps=10):
    total_steps = cue_steps + delay_steps + decision_steps
    X = torch.zeros(batch_size, total_steps, 3)
    labels = torch.zeros(batch_size, dtype=torch.long)

    for i in range(batch_size):
        base = torch.randint(4, 7, (1,)).item()
        diff = torch.randint(1, 3, (1,)).item()
        if torch.rand(1).item() > 0.5:
            left_count, right_count = base + diff, base
            labels[i] = 0
        else:
            left_count, right_count = base, base + diff
            labels[i] = 1

        left_times = torch.randperm(cue_steps)[:left_count]
        right_times = torch.randperm(cue_steps)[:right_count]
        X[i, left_times, 0] = 1.0
        X[i, right_times, 1] = 1.0

        noise = (torch.rand(delay_steps, 2) < 0.08).float()
        X[i, cue_steps : cue_steps + delay_steps, :2] = noise
        X[i, cue_steps + delay_steps :, 2] = 1.0

    return X, labels

In [23]:
def generate_batch_flash_memory(batch_size=64, num_steps=30):
    X = torch.zeros(batch_size, num_steps, 2)
    labels = torch.randint(0, 2, (batch_size,))
    for i in range(batch_size):
        X[i, :5, labels[i]] = (torch.rand(5) > 0.3).float()
        X[i] += (torch.rand(num_steps, 2) > 0.95).float()
    return X.clamp(0, 1), labels

In [29]:
# train

steps=600
lr=0.01

#input_size, decision_window, tau_beta = 2, 5, 50.0
#batch_fn = lambda: generate_batch_flash_memory(batch_size=64, num_steps=100)
input_size, decision_window, tau_beta = 3, 10, 300.0
batch_fn = lambda: generate_hard_tmaze_batch(batch_size=64, cue_steps=60, delay_steps=50, decision_steps=10)

model = Simple_RSNN(
    input_size=input_size,
    hidden_size=128,
    output_size=2,
    tau_soma=20.0,
    tau_beta=tau_beta,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

for step in range(steps):
    x, y = batch_fn()
    x, y = x, y

    optimizer.zero_grad()
    outputs = model(x)  # (batch_size, total_steps, 2)

    # Average readout membrane potential over the final decision window
    logits = outputs[:, -decision_window:, :].mean(dim=1)
    loss = loss_fn(logits, y)

    loss.backward()
    optimizer.step()

    if (step + 1) % 50 == 0:
        acc = (logits.argmax(dim=-1) == y).float().mean().item()
        print(f"Step {step+1:3d} | Loss: {loss.item():.4f} | Batch Acc: {acc*100:.1f}%")


Step  50 | Loss: 0.6976 | Batch Acc: 45.3%
Step 100 | Loss: 0.6929 | Batch Acc: 54.7%
Step 150 | Loss: 0.6910 | Batch Acc: 57.8%
Step 200 | Loss: 0.6945 | Batch Acc: 48.4%
Step 250 | Loss: 0.6932 | Batch Acc: 48.4%
Step 300 | Loss: 0.6969 | Batch Acc: 43.8%
Step 350 | Loss: 0.6924 | Batch Acc: 67.2%
Step 400 | Loss: 0.6932 | Batch Acc: 50.0%
Step 450 | Loss: 0.6914 | Batch Acc: 53.1%
Step 500 | Loss: 0.6942 | Batch Acc: 48.4%
Step 550 | Loss: 0.6892 | Batch Acc: 59.4%
Step 600 | Loss: 0.6896 | Batch Acc: 54.7%
